# CVE Details Fetcher - NVD API + Ollama

Fetch CVE details from NIST NVD API and optionally enhance with local LLM analysis.

In [ ]:
import json
import re
from typing import Any

import requests
from IPython.display import JSON, display


In [ ]:
# configuration
NVD_API = "https://services.nvd.nist.gov/rest/json/cves/2.0"
OLLAMA_API = "http://localhost:11434/v1/chat/completions"
OLLAMA_MODEL = "llama3.2"
USE_OLLAMA = False
API_TIMEOUT = 10


## Step 1: NVD Fetch & Parse

In [ ]:
def fetch_and_parse_cve(cve_id: str) -> dict[str, Any]:
    """Fetch a CVE from NVD and return a clean summary."""
    normalized_id = cve_id.strip().upper()

    if not re.fullmatch(r"CVE-\d{4}-\d{4,}", normalized_id):
        return {"error": f"Invalid CVE format: {cve_id}. Use: CVE-YYYY-XXXXX"}

    try:
        response = requests.get(f"{NVD_API}?cveId={normalized_id}", timeout=API_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        vulnerabilities = data.get("vulnerabilities", [])
        if not vulnerabilities:
            return {"error": f"{normalized_id} not found in NVD database"}

        cve = vulnerabilities[0]["cve"]
        metrics = cve.get("metrics", {})

        cvss_v3 = {}
        if "cvssMetricV31" in metrics:
            metric = metrics["cvssMetricV31"][0].get("cvssData", {})
            cvss_v3 = {"score": metric.get("baseScore"), "severity": metric.get("baseSeverity")}
        elif "cvssMetricV30" in metrics:
            metric = metrics["cvssMetricV30"][0].get("cvssData", {})
            cvss_v3 = {"score": metric.get("baseScore"), "severity": metric.get("baseSeverity")}

        cvss_v2 = {}
        if "cvssMetricV2" in metrics:
            cvss_v2 = {"score": metrics["cvssMetricV2"][0].get("cvssData", {}).get("baseScore")}

        cwe_list = []
        for weakness in cve.get("weaknesses", []):
            for item in weakness.get("description", []):
                value = item.get("value")
                if value:
                    cwe_list.append(value)

        references = [ref.get("url") for ref in cve.get("references", []) if ref.get("url")]

        affected_products = []
        for config in cve.get("configurations", []):
            for node in config.get("nodes", []):
                for match in node.get("cpeMatch", []):
                    criteria = match.get("criteria")
                    if criteria:
                        affected_products.append(criteria)

        return {
            "success": True,
            "cve_id": normalized_id,
            "description": cve.get("descriptions", [{}])[0].get("value"),
            "published": cve.get("published"),
            "cvss_v3": cvss_v3,
            "cvss_v2": cvss_v2,
            "cwe_list": cwe_list,
            "affected_products": affected_products[:10],
            "references": references[:5],
            "stats": {
                "total_products": len(affected_products),
                "total_cwes": len(cwe_list),
                "total_refs": len(references),
            },
        }

    except requests.exceptions.Timeout:
        return {"error": "NVD API timeout - check internet connection"}
    except requests.exceptions.ConnectionError:
        return {"error": "Cannot connect to NVD API"}
    except Exception as exc:
        return {"error": f"API error: {exc}"}


## Step 2: Ollama Analysis

In [ ]:
def ollama_security_report(cve_data: dict[str, Any]) -> dict[str, Any]:
    """Add a brief security summary from local Ollama if available."""
    if "error" in cve_data:
        return cve_data

    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
    except requests.RequestException:
        cve_data["ollama_status"] = "unavailable"
        cve_data["ollama_note"] = "Ollama not running. Install: https://ollama.ai | Run: ollama pull llama3.2"
        return cve_data

    try:
        description = cve_data.get("description", "")[:500]
        severity = cve_data.get("cvss_v3", {}).get("severity", "Unknown")

        prompt = (
            f"Analyze this CVE in 2-3 sentences:\n"
            f"CVE: {cve_data['cve_id']}\n"
            f"Severity: {severity}\n"
            f"Description: {description}...\n\n"
            "Provide: 1) what it is, 2) who is affected, 3) recommended action"
        )

        response = requests.post(
            OLLAMA_API,
            json={
                "model": OLLAMA_MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "stream": False,
            },
            timeout=30,
        )

        if response.status_code == 200:
            summary = response.json().get("choices", [{}])[0].get("message", {}).get("content", "")
            cve_data["ollama_analysis"] = {"model": OLLAMA_MODEL, "report": summary.strip()}
        else:
            cve_data["ollama_status"] = "error"

    except requests.RequestException:
        cve_data["ollama_status"] = "error"

    return cve_data


## Step 3: Automated Test Cases

In [ ]:
def test_cve_fetcher():
    """Run a small verification suite for valid and invalid CVE inputs."""
    tests = [
        ("CVE-2024-38063", True, "Valid CVE with Ollama", False),
        ("CVE-2021-44228", False, "Valid CVE without Ollama (Log4Shell)", False),
        ("cve-2024-38063", False, "Lowercase CVE id is normalized", False),
        ("CVE-INVALID-999", False, "Invalid format is rejected", True),
        ("CVE-1999-99999", False, "Non-existent CVE is rejected", True),
        ("", False, "Empty input is rejected", True),
    ]

    print("Running Test Suite")
    print("=" * 60)

    results = []
    for cve_id, use_ollama, description, expect_error in tests:
        print(f"\n[TEST] {description}")
        print(f"  CVE: {cve_id!r} | Ollama: {use_ollama}")

        data = fetch_and_parse_cve(cve_id)
        if use_ollama and "error" not in data:
            data = ollama_security_report(data)

        got_error = "error" in data
        passed = got_error == expect_error
        status = "PASS" if passed else "FAIL"
        print(f"  Result: {status}")

        if got_error:
            print(f"  Error: {data['error']}")
        else:
            print(f"  CVE ID: {data.get('cve_id')}")
            print(f"  Severity: {data.get('cvss_v3', {}).get('severity', 'N/A')}")

        results.append({"test": description, "cve": cve_id, "status": status, "data": data})

    print("\n" + "=" * 60)
    passed_count = sum(1 for result in results if result["status"] == "PASS")
    print(f"Tests Complete: {passed_count}/{len(results)} passed")
    return results


In [ ]:
# Run tests
test_results = test_cve_fetcher()

In [ ]:
print("\nFetching CVE Details")
print("=" * 60)

CVE_TO_FETCH = "CVE-2024-38063"
print(f"\nFetching: {CVE_TO_FETCH}...\n")

result = fetch_and_parse_cve(CVE_TO_FETCH)
if USE_OLLAMA:
    result = ollama_security_report(result)

if "error" not in result:
    print(f"Success: {result['cve_id']}")
    print(f"Severity: {result.get('cvss_v3', {}).get('severity', 'N/A')}")
    print(f"Score (v3): {result.get('cvss_v3', {}).get('score', 'N/A')}")
else:
    print(f"Error: {result['error']}")

print("\n" + "=" * 60)


In [ ]:
print("\nFull JSON Output:\n")
print(json.dumps(result, indent=2, default=str))
display(JSON(result))


In [ ]:
def save_results(data: dict[str, Any], filename: str | None = None) -> str | None:
    """Save CVE results to a JSON file."""
    if filename is None and "cve_id" in data:
        filename = f"{data['cve_id'].replace('-', '_')}_details.json"

    if not filename:
        return None

    with open(filename, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, default=str)

    print(f"Saved to: {filename}")
    return filename

# Uncomment to save:
# save_results(result)
